In [101]:
import pandas as pd
import numpy as np
import os
import sys
from io import BytesIO
from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
from pathlib import Path

import umap
import hdbscan

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score, confusion_matrix, classification_report

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy.optimize import linear_sum_assignment

pio.renderers.default = "vscode"

# ── Path helpers ──────────────────────────────────────────────────────────────
# Notebook lives in Modeling/GMM_testing/ → project root is two levels up
ROOT = Path(os.path.abspath("../.."))
sys.path.insert(0, str(ROOT))

BOILING_PLOTS_DIR = ROOT / "visuals" / "boiling_plots"
CSV_DIR           = ROOT / "data" / "CSV"
FEATURES_CSV      = ROOT / "data" / "non_noise_features.csv"

try:
    from visuals.visualization_fixed import visualize_csv_data as visualize_csv_data_fixed
    _HAS_VIZ = True
except ImportError:
    _HAS_VIZ = False
    print("Warning: visualization_fixed not found — click-to-plot disabled")


def image_path_for_file_name(file_name):
    if not isinstance(file_name, str) or not file_name.strip():
        return None
    base = file_name.strip().replace(".csv", "")
    exact = BOILING_PLOTS_DIR / (base + ".png")
    if exact.is_file():
        return str(exact)
    if not BOILING_PLOTS_DIR.is_dir():
        return None
    for f in os.listdir(BOILING_PLOTS_DIR):
        if not f.lower().endswith(".png"):
            continue
        if f.startswith(base) or base in f or f.replace(".png", "") == base:
            return str(BOILING_PLOTS_DIR / f)
    return None


def csv_path_for_file_name(file_name):
    if not isinstance(file_name, str) or not file_name.strip() or not CSV_DIR.is_dir():
        return None
    fn = file_name.strip()
    exact = CSV_DIR / fn
    if exact.is_file():
        return str(exact)
    for f in os.listdir(CSV_DIR):
        if not f.lower().endswith(".csv"):
            continue
        if f == fn or f.strip() == fn:
            return str(CSV_DIR / f)
    return None
from tqdm.notebook import tqdm

In [102]:
# ── Load data ─────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(FEATURES_CSV)
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")

Loaded 284 rows, 72 columns


In [103]:
# ── Preprocess ────────────────────────────────────────────────────────────────
file_names = df_raw["file_name"].reset_index(drop=True)
X = df_raw.drop(columns=["file_name"]).copy()

X = X.replace({"True": 1, "False": 0, True: 1, False: 0})
X = X.apply(pd.to_numeric, errors="coerce")

non_numeric_cols = X.columns[X.isna().all()].tolist()
if non_numeric_cols:
    print(f"Dropping non-numeric columns: {non_numeric_cols}")
    X = X.drop(columns=non_numeric_cols)

X = X.fillna(X.mean())
print(f"Feature matrix: {X.shape}")

Feature matrix: (284, 71)


In [104]:
# ── Scale + UMAP ──────────────────────────────────────────────────────────────
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X)

UMAP_PARAMS = dict(
    n_neighbors  = 15,
    min_dist     = 0.01,
    n_components = 3,
    metric       = "euclidean",
    random_state = 41,
)

umap_model = umap.UMAP(**UMAP_PARAMS)
X_umap     = umap_model.fit_transform(X_scaled)
print(f"UMAP embedding shape: {X_umap.shape}")

c:\Users\HELIOS-300\Desktop\NASA Capstone\NasaDataCapstone\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embedding shape: (284, 3)


In [105]:
# ── HDBSCAN ───────────────────────────────────────────────────────────────────
HDBSCAN_PARAMS = dict(
    min_cluster_size = 20,
    min_samples      = 5,
    metric           = "euclidean",
)

clusterer = hdbscan.HDBSCAN(**HDBSCAN_PARAMS)
labels    = clusterer.fit_predict(X_umap)

n_clusters  = len(set(labels)) - (1 if -1 in labels else 0)
n_noise     = (labels == -1).sum()
print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise} / {len(labels)}")
print(f"Label counts   : {dict(zip(*np.unique(labels, return_counts=True)))}")

Clusters found : 3
Noise points   : 0 / 284
Label counts   : {np.int64(0): np.int64(25), np.int64(1): np.int64(70), np.int64(2): np.int64(189)}


In [106]:
# ── Interactive 3D scatter ────────────────────────────────────────────────────
X_umap_arr   = np.asarray(X_umap)
n_components = X_umap_arr.shape[1]

plot_df = pd.DataFrame({
    "UMAP-1"   : X_umap_arr[:, 0],
    "UMAP-2"   : X_umap_arr[:, 1],
    "UMAP-3"   : X_umap_arr[:, 2] if n_components == 3 else 0.0,
    "cluster"  : labels.astype(str),
    "file_name": file_names.values,
})

color_seq      = px.colors.qualitative.Plotly
unique_clusters = sorted(plot_df["cluster"].unique(), key=lambda c: int(c))
color_map      = {c: color_seq[i % len(color_seq)] for i, c in enumerate(unique_clusters)}

traces = []
for cl in unique_clusters:
    sub   = plot_df[plot_df["cluster"] == cl]
    hover = "<b>%{customdata[0]}</b><extra>Cluster " + cl + "</extra>"
    traces.append(go.Scatter3d(
        x=sub["UMAP-1"], y=sub["UMAP-2"], z=sub["UMAP-3"],
        mode="markers",
        name=f"Cluster {cl}",
        marker=dict(size=3, color=color_map[cl]),
        customdata=sub[["file_name"]].to_numpy(),
        hovertemplate=hover,
    ))

fig_widget   = go.FigureWidget(
    data=traces,
    layout=go.Layout(
        title="UMAP + HDBSCAN — Full Dataset (GMM Testing Baseline)",
        width=900, height=700,
    ),
)
click_output = widgets.Output()

def _handle_click(trace, points, state, _out=click_output):
    with _out:
        _out.clear_output(wait=True)
        if not points.point_inds:
            return
        idx = points.point_inds[0]
        try:
            fn = trace.customdata[idx][0]
        except Exception:
            return
        print(f"Selected: {fn}")
        panels = []
        csv_p  = csv_path_for_file_name(fn)
        img_p  = image_path_for_file_name(fn)
        if csv_p and os.path.isfile(csv_p) and _HAS_VIZ:
            try:
                fig_fixed = visualize_csv_data_fixed(csv_p)
                buf = BytesIO()
                fig_fixed.savefig(buf, format="png", bbox_inches="tight", dpi=150)
                plt.close(fig_fixed)
                buf.seek(0)
                panels.append(widgets.Image(value=buf.read(), format="png",
                    layout=widgets.Layout(width="48%", object_fit="contain")))
            except Exception as e:
                print(f"Fixed-axis plot failed: {e}")
        if img_p and os.path.isfile(img_p):
            try:
                with open(img_p, "rb") as f:
                    panels.append(widgets.Image(value=f.read(), format="png",
                        layout=widgets.Layout(width="48%", object_fit="contain")))
            except Exception as e:
                print(f"Boiling plot failed: {e}")
        if panels:
            display(widgets.HBox(panels, layout=widgets.Layout(justify_content="space-around")))
        else:
            print(f"No images found for: {fn}")

for tr in fig_widget.data:
    tr.on_click(_handle_click)

display(widgets.VBox([fig_widget, click_output]))

    'data': [{'customdata': array([['MATLAB 1-51 PM Tue, Oct 15, 2024 Run0 .csv'…

In [107]:
# ── GMM experiment configuration ──────────────────────────────────────────────
TEST_SIZE_A1 = 0.10   # Approach 1: 90/10 split
TEST_SIZE_A2 = 0.05   # Approach 2: 95/5 split

# Approach 1: GMM on frozen UMAP — no UMAP refit, so 100 seeds is cheap (~seconds total)
NUM_SEEDS_A1 = 1000

# Approach 2: train-only pipeline — one full UMAP fit per seed, so keep lower
# 10 seeds ≈ 8 min  |  30 seeds ≈ 25 min  |  100 seeds ≈ 75 min
NUM_SEEDS_A2 = 250

# Number of non-noise HDBSCAN clusters found in the full-data run (cell 5)
# Used as fallback GMM n_components if train-only HDBSCAN finds 0 or 1 cluster
N_FULL_CLUSTERS = len(set(labels)) - (1 if -1 in labels else 0)
print(f"Full-data clusters : {N_FULL_CLUSTERS}")
print(f"Full-data noise    : {(labels == -1).sum()}")
print(f"A1 seeds           : {NUM_SEEDS_A1}")
print(f"A2 seeds           : {NUM_SEEDS_A2}")

Full-data clusters : 3
Full-data noise    : 0
A1 seeds           : 1000
A2 seeds           : 250


In [108]:
# ══════════════════════════════════════════════════════════════════════════════
# APPROACH 1 — GMM reproduces HDBSCAN labels in frozen UMAP space
# Loop over NUM_SEEDS_A1 different train/test splits; UMAP coordinates are fixed.
# ══════════════════════════════════════════════════════════════════════════════

# Filter noise once — GMM has no noise concept
mask_nn      = labels != -1
X1_umap_nn   = X_umap[mask_nn]
y1_nn        = labels[mask_nn]
n_gmm1       = len(np.unique(y1_nn))
print(f"Non-noise points : {mask_nn.sum()} / {len(labels)}  |  GMM components : {n_gmm1}\n")

a1_results = []   # list of dicts, one per seed

for seed in tqdm(range(1, NUM_SEEDS_A1 + 1), desc="Approach 1"):
    # Stratified split
    X1_tr, X1_te, y1_tr, y1_te = train_test_split(
        X1_umap_nn, y1_nn,
        test_size=TEST_SIZE_A1, random_state=seed, stratify=y1_nn,
    )

    # Fit GMM on train UMAP coordinates
    gmm = GaussianMixture(
        n_components=n_gmm1, covariance_type="full",
        random_state=seed, max_iter=500, n_init=3,
    )
    gmm.fit(X1_tr)

    # Predict + Hungarian alignment
    raw_pred     = gmm.predict(X1_te)
    unique_gmm   = np.unique(raw_pred)
    unique_hdb   = np.unique(y1_te)
    cost         = np.zeros((len(unique_gmm), len(unique_hdb)), dtype=int)
    for i, gc in enumerate(unique_gmm):
        for j, hc in enumerate(unique_hdb):
            cost[i, j] = np.sum((raw_pred == gc) & (y1_te == hc))
    ri, ci       = linear_sum_assignment(-cost)
    mapping      = {unique_gmm[r]: unique_hdb[c] for r, c in zip(ri, ci)}
    mapped_pred  = np.array([mapping[p] for p in raw_pred])

    ari = adjusted_rand_score(y1_te, mapped_pred)
    a1_results.append({"seed": seed, "ari": ari, "converged": gmm.converged_})

a1_df = pd.DataFrame(a1_results)
print(f"\nApproach 1 — ARI over {NUM_SEEDS_A1} seeds")
print(f"  Mean  : {a1_df['ari'].mean():.4f}")
print(f"  Std   : {a1_df['ari'].std():.4f}")
print(f"  Min   : {a1_df['ari'].min():.4f}")
print(f"  Max   : {a1_df['ari'].max():.4f}")
print(f"  Converged: {a1_df['converged'].sum()} / {NUM_SEEDS_A1}")

Non-noise points : 284 / 284  |  GMM components : 3



Approach 1:   0%|          | 0/1000 [00:00<?, ?it/s]


Approach 1 — ARI over 1000 seeds
  Mean  : 1.0000
  Std   : 0.0000
  Min   : 1.0000
  Max   : 1.0000
  Converged: 1000 / 1000


In [109]:
# ── Approach 1: Visualise ARI distribution ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(a1_df["ari"], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(a1_df["ari"].mean(), color="navy",   linewidth=2, linestyle="--",
           label=f"Mean = {a1_df['ari'].mean():.3f}")
ax.axvline(a1_df["ari"].mean() - a1_df["ari"].std(), color="cornflowerblue",
           linewidth=1.2, linestyle=":", label=f"±1 std")
ax.axvline(a1_df["ari"].mean() + a1_df["ari"].std(), color="cornflowerblue",
           linewidth=1.2, linestyle=":")
ax.set_xlabel("Adjusted Rand Index (ARI)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(
    f"Approach 1 — GMM vs HDBSCAN in Frozen UMAP Space\n"
    f"{NUM_SEEDS_A1} random splits  |  mean ARI = {a1_df['ari'].mean():.3f} ± {a1_df['ari'].std():.3f}",
    fontsize=12,
)
ax.legend(fontsize=11)
ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.savefig("approach1_ari_distribution.png", dpi=150)
plt.show()
print("Saved → approach1_ari_distribution.png")

Saved → approach1_ari_distribution.png


C:\Users\HELIOS-300\AppData\Local\Temp\ipykernel_46908\356569889.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [110]:
# ══════════════════════════════════════════════════════════════════════════════
# APPROACH 2 — True held-out generalization, train-only pipeline
# Each seed: new split → new scaler → new UMAP fit → new HDBSCAN → new GMM
# Test points projected via UMAP.transform() — they never influenced the embedding.
# ══════════════════════════════════════════════════════════════════════════════

idx_all    = np.arange(len(X))
a2_results = []

for seed in tqdm(range(1, NUM_SEEDS_A2 + 1), desc="Approach 2"):
    # 1. Split preprocessed feature matrix
    try:
        idx_tr, idx_te = train_test_split(
            idx_all, test_size=TEST_SIZE_A2, random_state=seed, stratify=labels,
        )
    except ValueError:
        idx_tr, idx_te = train_test_split(idx_all, test_size=TEST_SIZE_A2, random_state=seed)

    X_tr_df  = X.iloc[idx_tr].reset_index(drop=True)
    X_te_df  = X.iloc[idx_te].reset_index(drop=True)
    ref_labs = labels[idx_te]   # full-data HDBSCAN labels — reference only

    # 2. Scaler fit on train only
    sc2      = StandardScaler()
    X_tr_sc  = sc2.fit_transform(X_tr_df)
    X_te_sc  = sc2.transform(X_te_df)

    # 3. UMAP fit on train; project test via transform()
    um2      = umap.UMAP(**{**UMAP_PARAMS, 'random_state': seed})
    X_tr_um  = um2.fit_transform(X_tr_sc)
    X_te_um  = um2.transform(X_te_sc)

    # 4. HDBSCAN on train UMAP
    cl2        = hdbscan.HDBSCAN(**HDBSCAN_PARAMS)
    tr_labs    = cl2.fit_predict(X_tr_um)
    n_cl       = len(set(tr_labs)) - (1 if -1 in tr_labs else 0)
    n_noise_tr = (tr_labs == -1).sum()

    # Use full-data cluster count as fallback if train finds < 2 clusters
    n_components = max(n_cl, N_FULL_CLUSTERS) if n_cl >= 2 else N_FULL_CLUSTERS

    # 5. GMM on non-noise train UMAP points
    mask_tr_nn = tr_labs != -1
    if mask_tr_nn.sum() < n_components * 2:
        # Too few non-noise train points — skip this seed
        a2_results.append({
            "seed": seed, "ari": np.nan,
            "n_train_clusters": n_cl, "n_train_noise": n_noise_tr,
            "converged": False, "skipped": True,
        })
        continue

    gm2 = GaussianMixture(
        n_components=n_components, covariance_type="full",
        random_state=seed, max_iter=500, n_init=3,
    )
    gm2.fit(X_tr_um[mask_tr_nn])

    # 6. Predict test points; evaluate on non-noise reference points only
    raw2       = gm2.predict(X_te_um)
    mask_ref   = ref_labs != -1
    raw2_eval  = raw2[mask_ref]
    ref_eval   = ref_labs[mask_ref]

    if len(np.unique(ref_eval)) < 2:
        a2_results.append({
            "seed": seed, "ari": np.nan,
            "n_train_clusters": n_cl, "n_train_noise": n_noise_tr,
            "converged": gm2.converged_, "skipped": True,
        })
        continue

    # 7. Hungarian alignment
    u_gmm = np.unique(raw2_eval)
    u_ref = np.unique(ref_eval)
    cost2 = np.zeros((len(u_gmm), len(u_ref)), dtype=int)
    for i, gc in enumerate(u_gmm):
        for j, hc in enumerate(u_ref):
            cost2[i, j] = np.sum((raw2_eval == gc) & (ref_eval == hc))
    r2, c2  = linear_sum_assignment(-cost2)
    mapping2 = {u_gmm[r]: u_ref[c] for r, c in zip(r2, c2)}
    mapped2  = np.array([mapping2.get(p, -99) for p in raw2_eval])

    ari2 = adjusted_rand_score(ref_eval, mapped2)
    a2_results.append({
        "seed": seed, "ari": ari2,
        "n_train_clusters": n_cl, "n_train_noise": n_noise_tr,
        "converged": gm2.converged_, "skipped": False,
    })

a2_df      = pd.DataFrame(a2_results)
a2_valid   = a2_df.dropna(subset=["ari"])
print(f"\nApproach 2 — ARI over {len(a2_valid)} valid seeds ({a2_df['skipped'].sum()} skipped)")
print(f"  Mean  : {a2_valid['ari'].mean():.4f}")
print(f"  Std   : {a2_valid['ari'].std():.4f}")
print(f"  Min   : {a2_valid['ari'].min():.4f}")
print(f"  Max   : {a2_valid['ari'].max():.4f}")
print(f"\nTrain cluster count distribution:")
print(a2_df['n_train_clusters'].value_counts().sort_index())

Approach 2:   0%|          | 0/250 [00:00<?, ?it/s]

c:\Users\HELIOS-300\Desktop\NASA Capstone\NasaDataCapstone\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\HELIOS-300\Desktop\NASA Capstone\NasaDataCapstone\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\HELIOS-300\Desktop\NASA Capstone\NasaDataCapstone\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\HELIOS-300\Desktop\NASA Capstone\NasaDataCapstone\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\HELIOS-300\Desktop\NASA Capstone\NasaDataCapstone\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_stat


Approach 2 — ARI over 250 valid seeds (0 skipped)
  Mean  : 0.7563
  Std   : 0.2703
  Min   : 0.0634
  Max   : 1.0000

Train cluster count distribution:
n_train_clusters
2     60
3    160
4     16
5     11
6      3
Name: count, dtype: int64


In [111]:
# ── Approach 2: Visualise ARI distribution + train cluster stability ───────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: ARI histogram
ax = axes[0]
a2_valid = a2_df.dropna(subset=["ari"])
ax.hist(a2_valid["ari"], bins=15, color="tomato", edgecolor="white", alpha=0.85)
ax.axvline(a2_valid["ari"].mean(), color="darkred", linewidth=2, linestyle="--",
           label=f"Mean = {a2_valid['ari'].mean():.3f}")
ax.axvline(a2_valid["ari"].mean() - a2_valid["ari"].std(), color="salmon",
           linewidth=1.2, linestyle=":", label="±1 std")
ax.axvline(a2_valid["ari"].mean() + a2_valid["ari"].std(), color="salmon",
           linewidth=1.2, linestyle=":")
ax.set_xlabel("Adjusted Rand Index (ARI)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(
    f"Approach 2 — True Held-Out Generalization\n"
    f"{len(a2_valid)} seeds  |  mean ARI = {a2_valid['ari'].mean():.3f} ± {a2_valid['ari'].std():.3f}",
    fontsize=11,
)
ax.legend(fontsize=10)
ax.set_xlim(-0.05, 1.05)

# Right: how many clusters does train-only HDBSCAN find per seed?
ax2 = axes[1]
counts = a2_df["n_train_clusters"].value_counts().sort_index()
ax2.bar(counts.index.astype(str), counts.values, color="tomato", edgecolor="white", alpha=0.85)
ax2.axvline(str(N_FULL_CLUSTERS), color="darkred", linewidth=2, linestyle="--",
            label=f"Full-data clusters = {N_FULL_CLUSTERS}")
ax2.set_xlabel("Train-only HDBSCAN cluster count", fontsize=12)
ax2.set_ylabel("Seeds", fontsize=12)
ax2.set_title("Train Cluster Count Stability", fontsize=11)
ax2.legend(fontsize=10)

plt.suptitle("Approach 2 — Train-Only Pipeline Results", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("approach2_ari_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → approach2_ari_distribution.png")

# Side-by-side ARI comparison
print(f"\n{'':=<50}")
print(f"  Approach 1 (frozen UMAP)  :  {a1_df['ari'].mean():.3f} ± {a1_df['ari'].std():.3f}")
print(f"  Approach 2 (train-only)   :  {a2_valid['ari'].mean():.3f} ± {a2_valid['ari'].std():.3f}")
print(f"{'':=<50}")

Saved → approach2_ari_distribution.png

  Approach 1 (frozen UMAP)  :  1.000 ± 0.000
  Approach 2 (train-only)   :  0.756 ± 0.270


C:\Users\HELIOS-300\AppData\Local\Temp\ipykernel_46908\3048231741.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
